<a href="https://colab.research.google.com/github/Somtochukwu438/quizgen-llama3/blob/main/notebooks/01_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!nvidia-smi

Thu Oct 29 00:15:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
!pip -q install transformers peft bitsandbytes accelerate datasets
from google.colab import userdata, drive
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import os, sys
REPO = "/content/quizgen-llama3"
if os.path.exists(REPO):
    !git -C {REPO} pull
else:
    !git clone https://github.com/Somtochukwu438/quizgen-llama3.git {REPO}
sys.path.append(REPO)
from src.prompts import build_messages
from src.parse import parse_output

OUT = "/content/drive/MyDrive/quizgen"
os.makedirs(OUT, exist_ok=True)
print("Code loaded")

Cloning into '/content/quizgen-llama3'...
remote: Enumerating objects: 29, done.
remote: Counting objects: 100% (29/29), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 29 (delta 9), reused 9 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (29/29), 24.21 KiB | 12.11 MiB/s, done.
Resolving deltas: 100% (9/9), done.
Code loaded


In [3]:
from datasets import load_dataset
sciq = load_dataset("allenai/sciq")
test = sciq["test"].filter(lambda x: x["support"].strip() != "")
eval_set = test.shuffle(seed=42).select(range(300))
print(len(eval_set), "evaluation passages")

README.md:   0%|          | 0.00/7.02k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.99MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  339kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  343kB            

... (2 outputs left out)

Generating validation split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

300 evaluation passages


In [4]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL = "meta-llama/Llama-3.2-3B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
tok.pad_token = tok.eos_token
tok.padding_side = "left"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=bnb, device_map="auto")
model.eval()
print("Model loaded")

config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

Model loaded


In [9]:
def generate(passages, batch_size=8, max_new_tokens=200):
    outs = []
    for i in range(0, len(passages), batch_size):
        batch = passages[i : i + batch_size]
        prompts = [
            tok.apply_chat_template(build_messages(p), tokenize=False, add_generation_prompt=True)
            for p in batch
        ]
        enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        with torch.no_grad():
            gen = model.generate(**enc, max_new_tokens=max_new_tokens,
                                 do_sample=False, pad_token_id=tok.pad_token_id)
        outs += tok.batch_decode(gen[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)
        print(f"{len(outs)}/{len(passages)}")
    return outs

In [10]:
sample = generate(eval_set["support"][:3])
for out in sample:
    print("RAW:", out)
    print("PARSED:", parse_output(out))
    print("-" * 60)

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


3/3
RAW: {"question": "What is the estimated age at which the total loss of thymic epithelial tissue and thymocytes would occur if the rate of thymic involution continues at its current pace?", "answer": "120", "distractors": ["100", "150", "200"]}
PARSED: ({'question': 'What is the estimated age at which the total loss of thymic epithelial tissue and thymocytes would occur if the rate of thymic involution continues at its current pace?', 'answer': '120', 'distractors': ['100', '150', '200'], 'strict': True}, None)
------------------------------------------------------------
RAW: {"question": "How long does a typical biochemical reaction take without an enzyme?", "answer": "hours or days", "distractors": ["seconds", "minutes", "years"]}
PARSED: ({'question': 'How long does a typical biochemical reaction take without an enzyme?', 'answer': 'hours or days', 'distractors': ['seconds', 'minutes', 'years'], 'strict': True}, None)
------------------------------------------------------------


In [11]:
import json
from collections import Counter

raw = generate(eval_set["support"])

records = []
for ex, out in zip(eval_set, raw):
    parsed, err = parse_output(out)
    records.append({
        "passage": ex["support"],
        "ref_question": ex["question"],
        "ref_answer": ex["correct_answer"],
        "ref_distractors": [ex["distractor1"], ex["distractor2"], ex["distractor3"]],
        "raw": out,
        "parsed": parsed,
        "error": err,
    })

with open(f"{OUT}/baseline_outputs.jsonl", "w") as f:
    for r in records:
        f.write(json.dumps(r) + "\n")

n = len(records)
valid = sum(r["parsed"] is not None for r in records)
strict = sum(r["parsed"] is not None and r["parsed"]["strict"] for r in records)
print(f"Valid:  {valid}/{n} ({valid/n:.1%})")
print(f"Strict: {strict}/{n} ({strict/n:.1%})")
print("Errors:", Counter(r["error"] for r in records if r["error"]))

8/300
16/300
24/300
32/300
40/300
48/300
56/300
64/300
72/300
80/300
88/300
96/300
104/300
112/300
120/300
128/300
136/300
144/300
152/300
160/300
168/300
176/300
184/300
192/300
200/300
208/300
216/300
224/300
232/300
240/300
248/300
256/300
264/300
272/300
280/300
288/300
296/300
300/300
Valid:  287/300 (95.7%)
Strict: 287/300 (95.7%)
Errors: Counter({'answer_in_distractors': 5, 'no_json': 3, 'invalid_json': 3, 'bad_distractors': 2})


In [12]:
for r in records[:5]:
    print("PASSAGE:", r["passage"][:200], "...")
    print("REFERENCE Q:", r["ref_question"], "|", r["ref_answer"])
    if r["parsed"]:
        p = r["parsed"]
        print("MODEL Q:    ", p["question"], "|", p["answer"])
        print("DISTRACTORS:", p["distractors"])
    else:
        print("FAILED:", r["error"], "|", r["raw"][:200])
    print("=" * 70)

PASSAGE: Immune System By the year 2050, 25 percent of the population of the United States will be 60 years of age or older. The CDC estimates that 80 percent of those 60 years and older have one or more chron ...
REFERENCE Q: By the year 2050, 25 percent of the population of the united states will be 60 years of age or older. the cdc estimates that 80 percent of those 60 years and older have one or more chronic disease associated with deficiencies of this? | immune system
MODEL Q:     What is the estimated age at which the total loss of thymic epithelial tissue and thymocytes would occur if the rate of thymic involution continues at its current pace? | 120
DISTRACTORS: ['100', '150', '200']
PASSAGE: Enzymes are extremely efficient in speeding up biochemical reactions. They can catalyze up to several million reactions per second. As a result, the difference in rates of biochemical reactions with a ...
REFERENCE Q: Enzymes can catalyze up to several million reactions per what? | second


In [13]:
for r in records:
    if r["error"]:
        print(r["error"], "|", r["raw"][:300].replace("\n", " "))
        print("-" * 70)

answer_in_distractors | {"question": "What enzyme wraps around DNA to relieve tension during replication?", "answer": "topoisomerase", "distractors": ["topoisomerase", "DNA helicase", "DNA polymerase"]}
----------------------------------------------------------------------
no_json | {"question": "What is the relationship between the products of photosynthesis and cellular respiration?", "answer": "They are needed for each other", "distractors": ["The products of photosynthesis are the only source of energy for cellular respiration.", "The products of cellular respiration are th
----------------------------------------------------------------------
bad_distractors | {"question": "What is the primary characteristic of a tissue in an organism?", "answer": "a group of connected cells with a similar function", "distractors": ["a single cell with multiple functions", "a complex organ with multiple types of cells", "a type of cell with multiple functions", "a type of
-------------------------

In [14]:
for r in records:
    if r["error"] in ("no_json", "invalid_json"):
        n_tok = len(tok(r["raw"], add_special_tokens=False)["input_ids"])
        print(r["error"], "| tokens:", n_tok, "| ends with:", r["raw"][-80:].replace("\n", " "))

no_json | tokens: 86 | ends with: tosynthesis and cellular respiration are completely independent of each other."]
no_json | tokens: 114 | ends with: on.", "Blood will be redirected to the venous system through a vascular shunt."]
no_json | tokens: 65 | ends with: al and perform the same function.", "They are all connected to form a network."]
invalid_json | tokens: 97 | ends with:  have enough gravity to become round and are too small to have an atmosphere."}}
invalid_json | tokens: 76 | ends with:  power plants.", "In hot exhausts from vehicles, factories, and power plants."}}
invalid_json | tokens: 64 | ends with: ate the exchange of nutrients and waste", "to regulate the cell's temperature"}}


In [ ]:
!pip -q install transformers peft bitsandbytes accelerate datasets

In [ ]:
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))

In [ ]:
from datasets import load_dataset
sciq = load_dataset("allenai/sciq")
sciq = sciq.filter(lambda x: x["support"].strip() != "")
print({k: len(v) for k, v in sciq.items()})

ex = sciq["train"][0]
print(ex["support"][:300])
print("Q:", ex["question"])
print("A:", ex["correct_answer"])
print("Distractors:", [ex[f"distractor{i}"] for i in (1, 2, 3)])

README.md:   0%|          | 0.00/7.02k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.99MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  339kB            

data/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  343kB            

... (4 outputs left out)

Filter:   0%|          | 0/11679 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/1000 [00:00<?, ? examples/s]

{'train': 10481, 'validation': 887, 'test': 884}
Mesophiles grow best in moderate temperature, typically between 25°C and 40°C (77°F and 104°F). Mesophiles are often found living in or on the bodies of humans or other animals. The optimal growth temperature of many pathogenic mesophiles is 37°C (98°F), the normal human body temperature. Mesophilic
Q: What type of organism is commonly used in preparation of foods such as cheese and yogurt?
A: mesophilic organisms
Distractors: ['protozoa', 'gymnosperms', 'viruses']


In [ ]:
   from transformers import AutoTokenizer
   tok = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-3B-Instruct")
   print("Access OK")


config.json:   0%|          | 0.00/878 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Access OK
